# 🏦 Hybrid Intelligence AML Risk Scoring & SAR Narrative Generation
**Dataset:** SAML-D.csv (~1GB, ~9.5M rows) | **Target:** Google Colab (16GB RAM)

### Architecture Overview
| Layer | Component | Purpose |
|-------|-----------|--------|
| 0 | Memory-Safe Ingestion | Chunked CSV, stratified sampling |
| 1 | Statistical Features | Account-level behavioral aggregation |
| 2 | Graph Analytics | NetworkX: degree, PageRank, betweenness, cycles |
| 3 | Rule Engine | Deterministic AML compliance rules |
| 4 | Isolation Forest | Unsupervised anomaly scoring |
| 5 | Typology Mapper | Pattern→AML typology classification |
| 6 | Context Correlator | Linked accounts & transaction chains |
| 7 | SAR Generator | Template-RAG narrative (no hallucination) |
| 8 | Audit Trail | JSON evidence packages per flagged account |

In [ ]:
# ── Install any missing packages ──────────────────────────────────────────
!pip install -q networkx scikit-learn seaborn

In [ ]:
# ── Upload SAML-D.csv (run this cell in Colab) ────────────────────────────
# from google.colab import files
# uploaded = files.upload()   # select SAML-D.csv
# OR mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/SAML-D.csv'

DATA_PATH = 'SAML-D.csv'   # ← update path here
print('Data path set to:', DATA_PATH)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FULL SYSTEM CODE  –  paste aml_hybrid_system.py here OR run:
# exec(open('aml_hybrid_system.py').read())
# ════════════════════════════════════════════════════════════════════════════
exec(open('aml_hybrid_system.py').read())

In [ ]:
# ── Configure & Run Pipeline ──────────────────────────────────────────────
CHUNK_SIZE  = 250_000   # rows per chunk; reduce to 100_000 if OOM
MAX_CHUNKS  = None      # set to 10 for quick dev test

pipeline = AMLPipeline(data_path=DATA_PATH, output_dir=OUTPUT_DIR)
results, metrics = pipeline.run(top_k_sar=100)

In [ ]:
# ── View evaluation metrics ───────────────────────────────────────────────
import json
print(json.dumps(metrics, indent=2))

In [ ]:
# ── Display dashboard ─────────────────────────────────────────────────────
from IPython.display import Image
Image('aml_outputs/dashboard.png')

In [ ]:
# ── Inspect top SAR narratives ────────────────────────────────────────────
import pandas as pd
sar = pd.read_csv('aml_outputs/sar_narratives.csv')
for _, row in sar.head(3).iterrows():
    print('='*70)
    print(f"Account: {row['account_id']}  |  Score: {row['anomaly_score']:.4f}  |  Typology: {row['typology']}")
    print()
    print(row['narrative'])
    print()

In [ ]:
# ── Risk score leaderboard ────────────────────────────────────────────────
top10 = results.nlargest(10, 'anomaly_score')[[
    'account_id','anomaly_score','rule_score','typology','tx_count','amt_mean'
]]
print(top10.to_string(index=False))

In [ ]:
# ── Download all outputs ──────────────────────────────────────────────────
import shutil
shutil.make_archive('aml_outputs', 'zip', 'aml_outputs')
from google.colab import files
files.download('aml_outputs.zip')